<a href="https://colab.research.google.com/github/chaseshallow/ansci_4040_final_class_project/blob/main/Step02cSam2TrackCompare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 02c · Track with SAM 2.1: test (4 minutes)

ByteTrack (Step 02b) split one cow into 47 tracks, mostly while she lay down and strained. **SAM 2.1's video mode** follows the cow's *mask* using a memory of how she looks, so it keeps her as one track where YOLO loses her. This notebook tests it on **1080–1320 s** (4 minutes, ~5 min on a T4) before the full hour in **Step 02d**.

**How:** YOLO26n finds the cow at the start of each 2-minute chunk, SAM 2.1 Small follows her mask through the chunk, and chunks are joined where the boxes overlap. Every SAM box is compared with YOLO's boxes in the same frame:

- **Agrees** (`ok` in the preview): YOLO sees the cow in the same place
- **NoYoloBox** (`sam`): YOLO saw nothing, SAM kept following her
- **Disagrees** (`??`): YOLO sees a cow elsewhere but not where SAM is; SAM may have **drifted** (calf, floor): check the preview

**If it stops** (disconnect, GPU time used up): run cells 1–3 again with the same settings; finished chunks are kept in `Chunks/` and it continues from the next one.

Writes into the Shared Drive, `CalvingPenPipelineOutputs/<video>/Step02cSam2/`, **overwriting** each run:

- `Tracks.csv` (same columns as 02b, so step 03 can use it), `TrackSummary.csv`
- `Comparison.csv`: ByteTrack vs SAM 2 on the same time span
- `PreviewTracks.mp4`: masks + boxes + IDs, 4x speed
- `Chunks/`: per-chunk pieces, needed to resume

Needs a **T4 GPU**. The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install YOLO/SAM.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/ansci_4040_final_class_project.git"
RepoBranch = "main"
RepoDir = "/content/ansci_4040_final_class_project"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.Sam2Track import Sam2Track
from CalvingPipeline.Paths import OutputRoot
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. Settings: the only cell to edit.
VideoStem = "2026-09-26 00-00-00 Calving Pen 2"
VideoOutputDir = OutputRoot / VideoStem
ClipPath = VideoOutputDir / "ReferenceClip.mp4"                       # step 01
CompareTracksPath = VideoOutputDir / "Step02bFull" / "Tracks.csv"     # ByteTrack result to compare with (optional)
OutputDir = VideoOutputDir / "Step02cSam2"

StartSeconds = 1080    # seconds into the clip (add 03:35:11 for video time)
EndSeconds = 1320   # None = to the end of the clip
TargetFps = 2          # the camera only updates ~2 times a second, so more frames add little
ChunkSeconds = 120     # SAM 2 memory grows per frame; restart it from a YOLO box every 2 min
SeedConfidence = 0.4   # YOLO box needed to start following a cow
Preview = True         # False = no preview video (faster)
PreviewSpeed = 4
Resume = True          # True = continue a stopped run (same settings); False = start over

assert ClipPath.exists(), f"Run step 01 first: {ClipPath}"

In [ ]:
# 3. Run: track with SAM 2 (continues where a stopped run ended), compare with ByteTrack, show the preview.
import pandas as pd
from IPython.display import Video, display

Stats = Sam2Track(ClipPath, OutputDir, StartSeconds, EndSeconds, TargetFps=TargetFps, ChunkSeconds=ChunkSeconds,
                  SeedConfidence=SeedConfidence, CompareTracksPath=CompareTracksPath,
                  Preview=Preview, PreviewSpeed=PreviewSpeed, Resume=Resume)

display(pd.read_csv(OutputDir / "Comparison.csv"))
display(pd.read_csv(OutputDir / "TrackSummary.csv"))
PreviewPath = OutputDir / "PreviewTracks.mp4"
if PreviewPath.exists():
    PreviewMb = PreviewPath.stat().st_size / 2**20
    print(f"Preview: {PreviewPath} ({PreviewMb:.0f} MB)")
    if PreviewMb < 50:   # bigger files are too heavy to play inside the notebook; open them from Drive
        display(Video(str(PreviewPath), embed=True, width=960))

## About this step

**Models:** **YOLO26n** (detection, only to start and check) + **SAM 2.1 Small in video mode** (Ultralytics `SAM2VideoPredictor`, `sam2.1_s.pt`, ~46 M parameters) for following the cow. ByteTrack is not used.

**What we found (first run, 1080–1980 s at 5 fps, stopped after 45 min):** SAM kept the cow as **one track for all 900 s**; YOLO agreed with SAM's box on **97 %** of frames (median overlap 0.93). ByteTrack had broken the same stretch into many short tracks. So SAM replaces ByteTrack: the full hour runs in **Step 02d**, at 2 fps and resumable, because 5 fps was too slow (~2–3 frames/s).

**How to judge a result:** in `Comparison.csv`, fewer tracks, a higher `FramesWithCowShare` and longer tracks are better, **but only if** the preview shows the masks staying on the cow. A high `Disagrees` share, or masks that slide onto the calf or the floor, means SAM is drifting.

Full write-up, glossary and all models: `CalvingPen/README.md` in the repo.